# Phase 9-10 Data Quality Checkpoint

This notebook validates fixed outputs:
- missing values handling by type (text / categorical / numeric / date-like)
- cleaning correctness for `twcs_cleaned_fixed.csv`
- output integrity for next phases (`questions_for_ml_fixed.csv`, `conversations_for_rag.csv`, `labeled_questions_fixed.csv`)


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

RAW = Path("../data/raw/twcs.csv")
ENH = Path("../data/processed/twcs_enhanced.csv")
CLEAN = Path("../data/cleaned/twcs_cleaned_fixed.csv")
Q_ML = Path("../data/cleaned/questions_for_ml_fixed.csv")
RAG = Path("../data/cleaned/conversations_for_rag.csv")
LAB = Path("../data/labeled/labeled_questions_fixed.csv")

for p in [RAW, ENH, CLEAN, Q_ML, RAG, LAB]:
    assert p.exists(), f"Missing: {p.resolve()}"

print("All required files exist.")

All required files exist.


In [2]:
def count_rows(path: Path) -> int:
    # Robust count (safe with embedded newlines in text fields)
    total = 0
    for ch in pd.read_csv(path, usecols=[0], chunksize=250_000, low_memory=False):
        total += len(ch)
    return int(total)

counts = {
    "raw": count_rows(RAW),
    "enhanced": count_rows(ENH),
    "twcs_cleaned": count_rows(CLEAN),
    "questions_for_ml": count_rows(Q_ML),
    "conversations_for_rag": count_rows(RAG),
    "questions_labeled": count_rows(LAB),
}
pd.DataFrame({"dataset": counts.keys(), "rows": counts.values()})

,dataset,rows
0,raw,2811774
1,enhanced,2811774
2,twcs_cleaned,2811774
3,questions_for_ml,1537843
4,conversations_for_rag,2811774
5,questions_labeled,1537843


In [3]:
# Row consistency assertions
assert counts["raw"] == counts["enhanced"], "raw/enhanced mismatch"
assert counts["enhanced"] == counts["twcs_cleaned"], "enhanced/twcs_cleaned mismatch"
assert counts["enhanced"] == counts["conversations_for_rag"], "enhanced/rag mismatch"
assert counts["questions_for_ml"] == counts["questions_labeled"], "questions_for_ml/questions_labeled mismatch"
print("Row-count consistency checks passed.")

Row-count consistency checks passed.


In [4]:
# Missing-values profile by data type (on cleaned full table schema + sampled values)
sample = pd.read_csv(CLEAN, nrows=250_000, low_memory=False)
sample.columns = [c.strip() for c in sample.columns]

def classify_col(col, series):
    c = col.lower()
    if "text" in c:
        return "text"
    if "date" in c or "time" in c or c == "created_at":
        return "date_like"
    if pd.api.types.is_numeric_dtype(series):
        return "numeric"
    return "categorical"

rows = []
for col in sample.columns:
    s = sample[col]
    miss = s.isna().sum()
    empty = 0
    if s.dtype == object or pd.api.types.is_string_dtype(s):
        ss = s.fillna("").astype(str)
        empty = ss.str.strip().eq("").sum()
    rows.append({
        "column": col,
        "type_group": classify_col(col, s),
        "missing_na": int(miss),
        "missing_na_pct": round(100 * miss / len(sample), 3),
        "blank_string": int(empty),
        "blank_string_pct": round(100 * empty / len(sample), 3),
    })

mv = pd.DataFrame(rows).sort_values(["type_group", "missing_na_pct", "blank_string_pct"], ascending=[True, False, False])
mv

,column,type_group,missing_na,missing_na_pct,blank_string,blank_string_pct
11,response_tweet_ids,categorical,236235,94.494,236235,94.494
12,response_tweet_id,categorical,86728,34.691,86728,34.691
6,author_id,categorical,0,0.000,0,0.000
7,created_at,date_like,0,0.000,0,0.000
9,parent_tweet_id,numeric,88213,35.285,0,0.000
13,in_response_to_tweet_id,numeric,55018,22.007,0,0.000
0,tweet_id,numeric,0,0.000,0,0.000
1,conversation_id,numeric,0,0.000,0,0.000
2,position_in_conversation,numeric,0,0.000,0,0.000
3,total_tweets_in_conversation,numeric,0,0.000,0,0.000


In [5]:
# Cleaning-rule validation on full cleaned file (chunked)
url_probe = r"(?:https?://|t\.co/|\bwww\.)"
mention_probe = r"(?:^|[\s\n])@\w"

stats = {
    "rows": 0,
    "rag_url_residue": 0,
    "ml_url_residue": 0,
    "rag_mention_residue": 0,
    "ml_mention_residue": 0,
    "ml_uppercase_nonempty": 0,
    "ml_empty_placeholder": 0,
    "ml_blank_strings": 0,
}

for ch in pd.read_csv(CLEAN, usecols=["text_for_rag", "text_for_ml"], chunksize=250_000, low_memory=False):
    rag = ch["text_for_rag"].fillna("").astype(str)
    ml = ch["text_for_ml"].fillna("").astype(str)
    nonempty_ml = ml != "[EMPTY]"

    stats["rows"] += len(ch)
    stats["rag_url_residue"] += rag.str.contains(url_probe, case=False, regex=True, na=False).sum()
    stats["ml_url_residue"] += ml.str.contains(url_probe, case=False, regex=True, na=False).sum()
    stats["rag_mention_residue"] += rag.str.contains(mention_probe, regex=True, na=False).sum()
    stats["ml_mention_residue"] += ml.str.contains(mention_probe, regex=True, na=False).sum()
    stats["ml_uppercase_nonempty"] += ml[nonempty_ml].str.contains(r"[A-Z]", regex=True, na=False).sum()
    stats["ml_empty_placeholder"] += (ml == "[EMPTY]").sum()
    stats["ml_blank_strings"] += ml.str.strip().eq("").sum()

pd.DataFrame([stats]).T.rename(columns={0: "count"})

,count
rows,2811774
rag_url_residue,0
ml_url_residue,0
rag_mention_residue,0
ml_mention_residue,0
ml_uppercase_nonempty,0
ml_empty_placeholder,22459
ml_blank_strings,1


In [11]:
# Strong repair for hidden whitespace chars (ZWSP, BOM, NBSP, etc.)
import re
import pandas as pd

df_fix = pd.read_csv(CLEAN, low_memory=False, keep_default_na=False, na_filter=False)

def normalize_text(x: str) -> str:
    s = str(x)
    # remove invisible/problematic Unicode whitespace chars
    s = re.sub(r"[\u200B-\u200D\uFEFF\u00A0]", "", s)
    # collapse normal whitespace
    s = re.sub(r"\s+", " ", s).strip()
    return s

for col in ["text_for_ml", "text_for_rag"]:
    df_fix[col] = df_fix[col].astype(str).map(normalize_text)
    df_fix[col] = df_fix[col].mask(df_fix[col].eq(""), "[EMPTY]")

df_fix.to_csv(CLEAN, index=False)
print(f"✅ Re-saved cleaned file: {CLEAN.resolve()}")

# strict check
blank_ml = df_fix["text_for_ml"].astype(str).map(normalize_text).eq("").sum()
blank_rag = df_fix["text_for_rag"].astype(str).map(normalize_text).eq("").sum()
print("blank text_for_ml:", blank_ml)
print("blank text_for_rag:", blank_rag)

✅ Re-saved cleaned file: C:\projects\decision-assistant\data\cleaned\twcs_cleaned_fixed.csv
blank text_for_ml: 0
blank text_for_rag: 0


In [12]:
# Recompute key stats freshly from disk to avoid stale variable state.
url_probe = r"(?:https?://|t\.co/|\bwww\.)"
mention_probe = r"(?:^|[\s\n])@\w"

fresh = {
    "rag_url_residue": 0,
    "ml_url_residue": 0,
    "rag_mention_residue": 0,
    "ml_mention_residue": 0,
    "ml_uppercase_nonempty": 0,
    "ml_blank_strings": 0,
}

for ch in pd.read_csv(
    CLEAN,
    usecols=["text_for_rag", "text_for_ml"],
    chunksize=250_000,
    low_memory=False,
    keep_default_na=False,
    na_filter=False,
):
    rag = ch["text_for_rag"].astype(str)
    ml = ch["text_for_ml"].astype(str)
    nonempty_ml = ml != "[EMPTY]"

    fresh["rag_url_residue"] += rag.str.contains(url_probe, case=False, regex=True, na=False).sum()
    fresh["ml_url_residue"] += ml.str.contains(url_probe, case=False, regex=True, na=False).sum()
    fresh["rag_mention_residue"] += rag.str.contains(mention_probe, regex=True, na=False).sum()
    fresh["ml_mention_residue"] += ml.str.contains(mention_probe, regex=True, na=False).sum()
    fresh["ml_uppercase_nonempty"] += ml[nonempty_ml].str.contains(r"[A-Z]", regex=True, na=False).sum()
    fresh["ml_blank_strings"] += ml.str.strip().eq("").sum()

assert fresh["rag_url_residue"] == 0, f"RAG URL residues: {fresh['rag_url_residue']}"
assert fresh["ml_url_residue"] == 0, f"ML URL residues: {fresh['ml_url_residue']}"
assert fresh["rag_mention_residue"] == 0, f"RAG mention residues: {fresh['rag_mention_residue']}"
assert fresh["ml_mention_residue"] == 0, f"ML mention residues: {fresh['ml_mention_residue']}"
assert fresh["ml_uppercase_nonempty"] == 0, f"ML uppercase residues: {fresh['ml_uppercase_nonempty']}"
assert fresh["ml_blank_strings"] == 0, f"ML blank strings found: {fresh['ml_blank_strings']}"
print("Cleaning integrity assertions passed.")

Cleaning integrity assertions passed.


In [13]:
# Next-phase dataset checks
q_cols = pd.read_csv(Q_ML, nrows=0).columns.tolist()
r_cols = pd.read_csv(RAG, nrows=0).columns.tolist()
l_cols = pd.read_csv(LAB, nrows=0).columns.tolist()

assert q_cols == ["question_id", "question_text_for_ml", "conversation_id", "author_id", "created_at"], q_cols
assert r_cols == ["tweet_id", "conversation_id", "position", "text_for_rag", "author_id", "inbound"], r_cols
assert {"question_id", "question_text_for_ml", "conversation_id", "author_id", "created_at", "priority"}.issubset(set(l_cols)), l_cols

labs = pd.read_csv(LAB, usecols=["priority"], low_memory=False)
valid = set(labs["priority"].dropna().unique())
assert valid <= {"urgent", "normal"}, f"Invalid labels: {valid}"

print("Next-phase datasets are structurally correct.")
print(labs["priority"].value_counts(dropna=False).to_string())

Next-phase datasets are structurally correct.
priority
normal    1087531
urgent     450312


In [14]:
# Optional: skewness diagnostics before feature engineering (sample)
lab_sample = pd.read_csv(LAB, usecols=["question_text_for_ml"], nrows=200_000, low_memory=False)
txt = lab_sample["question_text_for_ml"].fillna("").astype(str)
feat = pd.DataFrame({
    "char_len": txt.str.len(),
    "word_count": txt.str.split().str.len(),
    "excl_count": txt.str.count("!"),
    "q_count": txt.str.count(r"\?"),
})

diag = feat.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T
diag["skew"] = feat.skew(numeric_only=True)
diag[["mean", "50%", "90%", "95%", "99%", "max", "skew"]]

,mean,50%,90%,95%,99%,max,skew
char_len,90.424185,89.0,145.0,199.0,266.0,296.0,0.773723
word_count,16.905515,16.0,28.0,37.0,50.0,66.0,0.800066
excl_count,0.250390,0.0,1.0,2.0,4.0,91.0,13.416713
q_count,0.326675,0.0,1.0,2.0,3.0,280.0,120.735099


## Missing values handling summary (what we did)

- **Text columns:**
  - `text_original`: preserved as-is (can still contain nulls from source).
  - `text_for_rag`: null -> empty string during processing, then mention/URL cleanup; kept as text.
  - `text_for_ml`: null/blank -> `[EMPTY]` placeholder after cleanup/lowercasing.

- **Categorical / ID-like columns** (`author_id`, `conversation_id`, etc.):
  - not imputed in cleaning phase; preserved as source truth.

- **Numerical columns:**
  - no synthetic imputation in Phase 9; values carried forward from enhanced dataset.

- **Date-like columns** (`created_at`):
  - preserved as raw string; parsing/feature extraction happens in Phase 11.
